## 需求：以加權指數自 2000/1/1 起日資料，當日收盤價若向上穿越過季線（60日），則買入；反之，則賣出。請統計出以下幾項資料：

- ⬤ 總損益、交易次數、勝率、平均賺賠比、最大連續虧損
- ⬤ 最後持有部位資訊（進場日、多空、價位）
- ⬤ （進階）統計所有出場交易之損益、持有時間

### P.S：
- 1）先以金融市場習慣說法為主，查了若仍不懂請勇於提問。利用每次機會知道對方的邏輯
- 2）本策略一旦進場後，非多即空，不會空手
- 3）可能是題組，請考量日後調整彈性
- 4）不限制使用工具（個人偏好EXCEL、GOOGLE SPREADSHEET）


# 加權指數季線策略假設

## 基本設定

- **標的**：台灣加權指數（TAIEX / Y9999）
- **資料頻率**：日資料
- **回測起始日**：2000/01/01
- **季線**：60 日簡單移動平均線（SMA60）

## 交易規則

### 做多

當：

- 昨日收盤價 ≤ 昨日 SMA60
- 今日收盤價 > 今日 SMA60

視為**向上穿越季線**，建立多頭部位（Long）。

### 做空

當：

- 昨日收盤價 ≥ 昨日 SMA60
- 今日收盤價 < 今日 SMA60

視為**向下穿越季線**，建立空頭部位（Short）。

### 部位規則

- 策略開始持有部位後，維持**非多即空**。
- 不存在空手狀態。
- 出現反向訊號時：
  1. 將原有部位平倉。
  2. 同時建立反向部位。

## 成交假設

- 訊號以**當日收盤價**判斷。
- 假設以**訊號當日收盤價成交**。

## 損益定義

- 損益先以**加權指數點數**表示。
- 暫不考慮：
  - 初始本金
  - 每點價值
  - 交易成本
  - 手續費
  - 稅費
  - 滑價

## 交易定義

一筆完整交易定義為：

**進場 → 持有 → 出場**

反手時：

- 原部位完成一筆交易。
- 同時開始下一筆反向交易。

尚未出場的最後一筆部位視為**未平倉部位（Open Position）**。

## 持有時間

持有時間以**交易日（Trading Days）**計算。

定義：

> 今日收盤進場，下一個交易日收盤出場，持有時間為 1 個交易日。

不使用日曆日計算，因此週末及休市日不額外計入持有時間。

## 績效指標

### 總損益

所有已完成交易之損益點數加總。

### 交易次數

完成「進場 → 出場」的交易筆數。

未平倉的最後部位不列入已完成交易次數。

### 勝率

獲利交易次數占全部已完成交易次數的比例。

### 平均賺賠比

平均獲利交易的獲利點數，相對於平均虧損交易虧損點數絕對值的比例。

### 最大連續虧損

歷史交易紀錄中，連續出現虧損交易的最大筆數。

## 最後持有部位

紀錄回測資料截止日仍持有之部位：

- 進場日
- 多空方向（Long / Short）
- 進場價位

## 出場交易明細

每筆已完成交易至少紀錄：

- 進場日
- 出場日
- 多空方向
- 進場價
- 出場價
- 損益點數
- 持有交易日數


## 第一筆部位如何建立
  - 等待回測起始日後第一次穿越訊號再進場

In [1]:
import pandas as pd
import os
from dotenv import load_dotenv
import psycopg

load_dotenv("../../.env")

try:
    conn = psycopg.connect(
        host=os.getenv("DB_HOST"),
        port=os.getenv("DB_PORT"),
        dbname=os.getenv("DB_NAME"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        connect_timeout=5
    )

    with conn.cursor() as cur:
        cur.execute("""
            SELECT
                current_database(),
                current_user,
                version();
        """)

        database, user, version = cur.fetchone()

    print("PostgreSQL 連線成功")
    print("Database :", database)
    print("User     :", user)
    print("Version  :", version)

except psycopg.Error as e:
    print("PostgreSQL 連線失敗")
    print(e)

finally:
    if 'conn' in locals():
        conn.close()

PostgreSQL 連線成功
Database : market_data
User     : market_data_user
Version  : PostgreSQL 18.4 (Debian 18.4-1.pgdg13+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit
